# 基于KNN的语音数据分类

在终端穿戴场景中，往往涉及到用户和终端设备之间的语音交互，通常由用户唤醒相关的设备并进行语音命令控制设备完成相应的操作，设备在收到语音命令时会对用户的意图进行识别，例如识别“拍照”“摄像”等意图，从而进行对应的操作。

要求使用KNN算法对用户输出的语音特征进行分类，相关规则与输入输出要求如下：

1. 语音片段经过特征提取后是一个3维的向量；

2. 所有距离使用欧式距离公式计算，即对于两个N维向量 $\vec{x} = (x_1, x_2, \dots, x_N), \quad \vec{y} = (y_1, y_2, \dots, y_N)$ ，两个向量间的距离为： $d_{(\vec{x},\vec{y})} = \sqrt{\sum_{k=1}^N (x_k - y_k)^2}$ 。

### 输入描述

第一行：正整数 $N$ 和 $K$ ，分别表示给定的已知语音特征向量的数量和邻居数量 $K$

第2~ $N+1$ 行：已知语音的特征向量和label，其中最后一位表示类别label

第 $N+2$ 行：待分类的语音特征向量

### 输出描述

给出待分类语音的类别，该值为正整数且在输入给定范围内。

### 样例1

**输入**

```Plain Text
10 3
0.5 0.3 0.4 0
0.6 0.2 0.5 0
0.4 0.3 0.3 0
0.7 0.4 0.6 0
2.1 2.3 2.2 1
2.3 2.2 2.4 1
2.2 2.4 2.3 1
4.5 4.3 4.4 2
4.4 4.5 4.6 2
4.6 4.4 4.5 2
2.2 2.1 2.3
```

**输出**

```Plain Text
1
```

**说明**

待分类的语音特征向量为 $(2.2, 2.1, 2.3)$ 。计算其与所有已知向量的欧式距离后，最近的3个邻居分别是：

 $(2.1, 2.3, 2.2)$  类别1

 $(2.3, 2.2, 2.4)$  类别1

 $(2.2, 2.4, 2.3)$  类别1

这3个邻居都属于类别1，因此待分类向量被判定为类别1。

### 样例2

**输入**

```Plain Text
12 4
1 0.9 1 1
0.8 0.9 0.7 1
1.3 1 1.2 1
1.2 0.9 1 1
2 2.2 2.1 2
2.3 2.2 2 2 2
2 2.2 1.9 2
1.9 2.2 2.1 2
3.1 3.1 3 3
2.8 2.9 3.1 3
2.9 3 3.2 3
3.1 3 3.1 3
2.2 1.2 1.9
```

**输出**

```Plain Text
2
```

**说明**

与 $2.2, 1.2, 1.9$ 最近的4个邻居中，类别最多的是2，故被分类为2。

In [4]:
import math
import numpy as np
class KDNode:
    def __init__(self, feature, label=None, split_dim=None, left=None, right=None):
        """
        KD树节点定义
        :param feature:样本的K维特征向量
        :param label: 样本的类别标签
        :param split_dim: 当前节点的划分维度
        :param left: 左子树
        :param right: 右子树
        """
        self.feature = feature
        self.label = label
        self.split_dim = split_dim
        self.left = left
        self.right = right

class KDTree:
    def __init__(self,features,labels=None):
        """
        KD树初始化与构造
        :param features: 样本特征集，shape为(n_samples,n_dim)
        :param labels: 样本标签集，shape为(n_sammples,)，可选（无监督场景可不用）
        """
        self.n_dim = features.shape[1] # 特征维度
        self.labels = labels if labels is not None else [None]*len(features)
        # 递归构建KD树
        self.root = self._build_tree(features,self.labels,depth=0)

    def _choose_split_dim(self,features,depth):
        """
        选择划分维度，两种策略可选：
        1.循环轮换维度：按深度取模维度
        2.最大方差维度：计算各维度方差，选最大的（高纬度场景）
        """
        # 策略1:循环轮换维度
        return depth % self.n_dim

    def _build_tree(self,features,labels,depth):
        """
        递归构建KD树的核心函数
        :param features: 当前待划分的样本特征集
        :param labels: 当前待划分的样本标签集
        :param depth: 当前树的深度，用于选择划分维度
        :return: 当前子树的根节点
        """
        # 递归终止条件
        if len(features) == 0:
            return None

        # 步骤1:选择当前划分维度
        split_dim = self._choose_split_dim(features,depth)

        # 步骤2:按划分维度排序，选择中位数节点作为划分点
        # 按划分维度对样本进行升序排序，获取排序索引
        sorted_indices = np.argsort(features[:,split_dim])
        median_idx = len(features)//2 # 中位数索引
        median_sample_idx = sorted_indices[median_idx]

        # 步骤3:创建当前节点
        current_node = KDNode(
            feature=features[median_sample_idx],
            label=labels[median_sample_idx],
            split_dim=split_dim
        )

        # 步骤4:递归构建左右子树
        # 左子树：划分维度上数值小于中位数的样本
        left_features = features[sorted_indices[:median_idx]]
        left_labels = [labels[i] for i in sorted_indices[:median_idx]]
        current_node.left = self._build_tree(left_features, left_labels, depth + 1)

        # 右子树：划分维度上数值大于等于中位数节点的样本
        right_features = features[sorted_indices[median_idx+1:]]
        right_labels = [labels[i] for i in sorted_indices[median_idx+1:]]
        current_node.right = self._build_tree(right_features, right_labels, depth + 1)

        return current_node

    def _cal_euclidean_distance(self,vec1,vec2):
        """计算两个向量的欧氏距离"""
        sum = 0.0
        i = 0
        while i < self.n_dim:
            sum += math.sqrt((vec1[i]-vec2[i])**2)
            i += 1
        return sum

    def _knn_search(self,node,target,k,nearest_list):
        """
        递归实现k近邻搜索核心逻辑
        :param node: 当前便利节点
        :param target: 待搜索的目标向量
        :param k: 近邻数量
        :param nearest_list:存储当前最近邻的列表，格式为[(距离,特征,标签),...]
        :return:
        """
        if node is None:
            return

        # 计算当前节点与目标向量的距离
        current_dist = self._cal_euclidean_distance(node.feature, target)

        # 更新最近邻列表
        if len(nearest_list) < k:
            nearest_list.append((current_dist, node.feature, node.label))
            # 按距离升序排序
            nearest_list.sort(key=lambda x: x[0])
        else:
            # 若当前距离小于列表中最大的距离，替换并重新排序
            if current_dist < nearest_list[-1][0]:
                nearest_list[-1] = (current_dist, node.feature, node.label)
                nearest_list.sort(key=lambda x: x[0])

        # 确定遍历顺序：先进入目标向量所在的子空间
        split_dim = node.split_dim
        if target[split_dim] < node.feature[split_dim]:
            # 目标在左子空间，先遍历左子树
            self._knn_search(node.left, target, k, nearest_list)
            # 剪枝判断：若目标到分割超平面的距离小于当前最大近邻距离，需遍历右子树
            if abs(target[split_dim] - node.feature[split_dim]) < nearest_list[-1][0]:
                self._knn_search(node.right, target, k, nearest_list)
        else:
            # 目标在右子空间，先遍历右子树
            self._knn_search(node.right, target, k, nearest_list)
            # 剪枝判断
            if abs(target[split_dim] - node.feature[split_dim]) < nearest_list[-1][0]:
                self._knn_search(node.left, target, k, nearest_list)

    def knn_predict(self, target ,k):
        """
        对外暴露的k近邻预测接口
        :param target: 待分类的特征向量
        :param k: 近邻数量
        :return: 最近邻列表、预测的类别标签
        """
        nearest_list = []
        self._knn_search(self.root, target, k, nearest_list)

        # 多数投票法确定分类结果（与题目规则完全一致）
        label_count = {}
        for item in nearest_list:
            label = item[2]
            label_count[label] = label_count.get(label, 0) + 1
        predict_label = max(label_count.items(), key=lambda x: x[1])[0]

        return nearest_list, predict_label

if __name__ == '__main__':
    # 样例1的已知语音特征集与标签集
    features = np.array([
        [0.5, 0.3, 0.4],
        [0.6, 0.2, 0.5],
        [0.4, 0.3, 0.3],
        [0.7, 0.4, 0.6],
        [2.1, 2.3, 2.2],
        [2.3, 2.2, 2.4],
        [2.2, 2.4, 2.3],
        [4.5, 4.3, 4.4],
        [4.4, 4.5, 4.6],
        [4.6, 4.4, 4.5]
    ])
    labels = np.array([0, 0, 0, 0, 1, 1, 1, 2, 2, 2])

    # 1. 构建KD树
    kd_tree = KDTree(features, labels)
    print("KD树构建完成！")

    # 2. 待分类的语音特征向量（样例1的输入）
    target_feature = np.array([2.2, 2.1, 2.3])
    k = 3  # 近邻数量，与样例1一致

    # 3. KNN近邻搜索与分类预测
    nearest_neighbors, predict_label = kd_tree.knn_predict(target_feature, k)

    # 4. 输出结果（与样例1的说明完全一致）
    print("\n===== 近邻搜索结果 =====")
    for idx, item in enumerate(nearest_neighbors):
        dist, feature, label = item
        print(f"第{idx+1}近邻：特征{feature}，类别{label}，欧氏距离{dist:.4f}")

    print(f"\n===== 分类结果 =====")
    print(f"待分类语音的类别为：{predict_label}")

KD树构建完成！

===== 近邻搜索结果 =====
第1近邻：特征[2.2 2.4 2.3]，类别1，欧氏距离0.3000
第2近邻：特征[2.3 2.2 2.4]，类别1，欧氏距离0.3000
第3近邻：特征[2.1 2.3 2.2]，类别1，欧氏距离0.4000

===== 分类结果 =====
待分类语音的类别为：1


In [5]:
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

# ===================== 1. 准备数据（与题目样例1完全一致）=====================
# 已知语音特征集（10个样本，每个样本3维特征）
X_train = np.array([
    [0.5, 0.3, 0.4],
    [0.6, 0.2, 0.5],
    [0.4, 0.3, 0.3],
    [0.7, 0.4, 0.6],
    [2.1, 2.3, 2.2],
    [2.3, 2.2, 2.4],
    [2.2, 2.4, 2.3],
    [4.5, 4.3, 4.4],
    [4.4, 4.5, 4.6],
    [4.6, 4.4, 4.5]
])

# 已知语音类别标签
y_train = np.array([0, 0, 0, 0, 1, 1, 1, 2, 2, 2])

# 待分类的语音特征向量（样例1的输入）
X_test = np.array([[2.2, 2.1, 2.3]])

# ===================== 2. 创建并训练KNN模型 =====================
# 参数说明：
# n_neighbors=3：对应题目中的K=3，选择3个最近邻
# metric='euclidean'：指定使用欧氏距离，与题目公式完全一致
knn = KNeighborsClassifier(n_neighbors=3, metric='euclidean')

# 训练模型（对于KNN，训练阶段只是存储数据，无实际训练过程）
knn.fit(X_train, y_train)

# ===================== 3. 预测并输出结果 =====================
# 预测待分类向量的类别
y_pred = knn.predict(X_test)

# 也可以获取最近邻的详细信息（距离和索引），对应样例1的说明
neighbor_distances, neighbor_indices = knn.kneighbors(X_test)

# ===================== 4. 打印结果 =====================
print("===== 最近邻详细信息 =====")
for i in range(3):
    idx = neighbor_indices[0][i]
    dist = neighbor_distances[0][i]
    print(f"第{i+1}近邻：特征{X_train[idx]}，类别{y_train[idx]}，欧氏距离{dist:.4f}")

print(f"\n===== 分类结果 =====")
print(f"待分类语音的类别为：{y_pred[0]}")

===== 最近邻详细信息 =====
第1近邻：特征[2.3 2.2 2.4]，类别1，欧氏距离0.1732
第2近邻：特征[2.1 2.3 2.2]，类别1，欧氏距离0.2449
第3近邻：特征[2.2 2.4 2.3]，类别1，欧氏距离0.3000

===== 分类结果 =====
待分类语音的类别为：1
